# Nivel II: Red Neuronal Espacio-Temporal sobre Grafos (ST-GNN + Binomial Negativa)

En el Nivel I vimos que **Auto-ARIMAX** supera a GLM y HGB, pero tiene dos limitaciones:
1. Modela cada cantón de forma aislada (no aprovecha brotes tempranos en cantones **CONECTADOS**).
2. Presenta un ligero desfase de aprox 1 semana en la subida y bajada de picos.

## Punto de Análisis

- **Grafo funcional $W \in \mathbb{R}^{82 \times 82}$** construido únicamente sobre `train <= 2023` (top-$k$ vecinos por correlación espacial, listo para sustituirse por matriz geográfica cuando se agregue el shapefile).
- **Arquitectura:** Convolución sobre Grafos (**GCN**) + **GRU temporal** + **conexión residual autorregresiva** + **cabeza probabilística Binomial Negativa (NB-NLL)**.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import mlflow

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
os.chdir(ROOT_DIR)

df = pd.read_csv(ROOT_DIR / "data/processed/dengue_panel_clean.csv")
df = df.sort_values(["time_idx", "canton"]).reset_index(drop=True)
cantons = np.sort(df["canton"].unique())
N_NODES = len(cantons)
T_TOTAL = df["time_idx"].nunique()
print(f"Tensor espacio-temporal base: T={T_TOTAL} semanas, N={N_NODES} cantones")

## 1. Construcción del Grafo Funcional $W$ (solo en `train <= 2023`)

Construimos una matriz de adyacencia esparsa seleccionando para cada cantón sus **$k = 8$ vecinos con mayor correlación positiva** en `log1p(casos)` durante `train`, la simetrizamos y aplicamos normalización espectral $\hat{A} = \tilde{D}^{-1/2} (W + I) \tilde{D}^{-1/2}$.

In [ ]:
train_only = df[df["year"] <= 2023].copy()
train_only["log_casos"] = np.log1p(train_only["casos"])
piv_tr = train_only.pivot(index="time_idx", columns="canton", values="log_casos")[cantons]
corr_tr = piv_tr.corr().fillna(0.0).to_numpy(dtype=np.float32)
np.fill_diagonal(corr_tr, 0.0)

TOP_K = 8
W = np.zeros_like(corr_tr)
for i in range(N_NODES):
    top_idx = np.argsort(corr_tr[i])[-TOP_K:]
    for j in top_idx:
        if corr_tr[i, j] > 0:
            W[i, j] = corr_tr[i, j]

# Simetrizar y agregar auto-conexiones (I)
W = np.maximum(W, W.T)
A_tilde = W + np.eye(N_NODES, dtype=np.float32)
deg = A_tilde.sum(axis=1)
d_inv_sqrt = np.power(deg, -0.5)
A_norm = (d_inv_sqrt[:, None] * A_tilde) * d_inv_sqrt[None, :]
A_norm_t = torch.tensor(A_norm, dtype=torch.float32, device=device)

print(f"Aristas no nulas (sin diagonal): {int((W > 0).sum())} | Grado promedio: {(W > 0).sum(axis=1).mean():.2f}")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(A_norm, cmap="viridis", ax=ax)
ax.set_title(f"Matriz de Adyacencia Normalizada A_norm (Top-K={TOP_K}, Train<=2023)")
fig.tight_layout()
plt.show()

## 2. Tensores, normalización y Dataset de ventanas deslizantes

Usamos una ventana de entrada de **$T_{\text{in}} = 12$ semanas** (respaldada por el ACF) para predecir `casos` en la semana $t$.  
Todas las covariables se estandarizan usando **únicamente media y desviación estándar de `train <= 2023`**.

In [ ]:
feature_cols = [
    "log_casos",
    "temp_prom", "tmin_mean", "precip_total", "ndvi", "nino34ssta",
    "dengue", "week_sin", "week_cos", "urb",
]

df["log_casos"] = np.log1p(df["casos"])
tr_mask_rows = df["year"] <= 2023

# Estandarizar covariables exógenas solo con estadísticas de Train (dejamos log_casos, week_sin, week_cos sin escalar)
scale_cols = ["temp_prom", "tmin_mean", "precip_total", "ndvi", "nino34ssta", "dengue", "urb"]
df_scaled = df.copy()
for c in scale_cols:
    mu = df.loc[tr_mask_rows, c].mean()
    sd = df.loc[tr_mask_rows, c].std() + 1e-6
    df_scaled[c] = (df[c] - mu) / sd

# Construir arreglo (T_TOTAL, N_NODES, F) y target (T_TOTAL, N_NODES)
X_all = np.zeros((T_TOTAL, N_NODES, len(feature_cols)), dtype=np.float32)
for f_i, col in enumerate(feature_cols):
    X_all[:, :, f_i] = df_scaled.pivot(index="time_idx", columns="canton", values=col)[cantons].to_numpy(dtype=np.float32)

Y_all = df.pivot(index="time_idx", columns="canton", values="casos")[cantons].to_numpy(dtype=np.float32)
years_by_t = df.groupby("time_idx")["year"].first().sort_index().to_numpy()

WINDOW = 12

class SpatioTemporalWindowDataset(Dataset):
    def __init__(self, X, Y, target_indices, window=12):
        self.X = X
        self.Y = Y
        self.indices = target_indices
        self.window = window

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        x_win = self.X[t - self.window : t]  # (W, N, F)
        y_t = self.Y[t]                      # (N,)
        return torch.from_numpy(x_win), torch.from_numpy(y_t)

tr_t = np.where((np.arange(T_TOTAL) >= WINDOW) & (years_by_t <= 2023))[0]
va_t = np.where(years_by_t == 2024)[0]
te_t = np.where(years_by_t == 2025)[0]

train_ds = SpatioTemporalWindowDataset(X_all, Y_all, tr_t, WINDOW)
val_ds = SpatioTemporalWindowDataset(X_all, Y_all, va_t, WINDOW)
test_ds = SpatioTemporalWindowDataset(X_all, Y_all, te_t, WINDOW)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f"Ventanas -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

## 3. Arquitectura ST-GNN + Función de Pérdida Binomial Negativa (NB-NLL)

Cada paso temporal $t$ pasa por:
1. **Capa GCN espacial:** $\text{ReLU}(\hat{A} X_t W_{\text{gcn}})$ para propagar señales entre cantones conectados.
2. **GRU temporal compartida por nodo:** procesa la secuencia de las $T_{\text{in}} = 12$ semanas por cantón.
3. **Cabeza Binomial Negativa con conexión residual:**
   - Predice $\mu_{it} = \exp(\log(1 + y_{i,t-1}) + \Delta_{it}) - 1$ (acotado positivamente vía `softplus` para estabilidad numérica), de modo que parte de la inercia local y usa la GNN+clima para anticipar subidas y bajadas.
   - Aprende un parámetro de dispersión $\alpha_i > 0$ por cantón.
   - Minimiza la **Negative Log-Likelihood de la Binomial Negativa**.

In [ ]:
class STGNNNegBin(nn.Module):
    def __init__(self, in_feats, hidden_dim=64, num_nodes=82, emb_dim=16, dropout=0.2):
        super().__init__()
        self.node_emb = nn.Parameter(torch.randn(num_nodes, emb_dim) * 0.05)
        self.e1 = nn.Parameter(torch.randn(num_nodes, 10) * 0.1)
        self.e2 = nn.Parameter(torch.randn(num_nodes, 10) * 0.1)

        in_total = in_feats + emb_dim
        self.self_proj = nn.Linear(in_total, hidden_dim)
        self.graph_proj1 = nn.Linear(in_total, hidden_dim)
        self.graph_proj2 = nn.Linear(hidden_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.head_delta = nn.Linear(hidden_dim, 1)
        nn.init.zeros_(self.head_delta.weight)
        nn.init.zeros_(self.head_delta.bias)

        # Parámetro de dispersión r = 1/alpha por cantón (en escala log)
        self.log_r = nn.Parameter(torch.zeros(num_nodes))

    def forward(self, x_win, a_norm):
        # x_win: (B, W, N, F)
        B, W, N, _ = x_win.shape
        last_log_y = x_win[:, -1, :, 0]  # log1p(casos_{t-1}) -> (B, N)

        # Adyacencia híbrida: grafo funcional fijo + adyacencia adaptativa aprendida
        a_adapt = F.softmax(F.relu(self.e1 @ self.e2.T), dim=-1)
        a_mix = 0.7 * a_norm + 0.3 * a_adapt

        emb = self.node_emb.unsqueeze(0).unsqueeze(0).expand(B, W, N, -1)
        x_cat = torch.cat([x_win, emb], dim=-1)

        # 1. Proyección propia + difusión espacial en grafo
        h_self = F.relu(self.self_proj(x_cat))
        h_nei = F.relu(self.graph_proj1(torch.einsum("ij,btjf->btif", a_mix, x_cat)))
        h = self.drop(h_self + h_nei)
        h = h + F.relu(self.graph_proj2(torch.einsum("ij,btjf->btif", a_mix, h)))

        # 2. GRU temporal por nodo: reordenamos a (B*N, W, H)
        h_seq = h.permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_last = self.gru(h_seq)
        h_last = self.drop(h_last.squeeze(0).reshape(B, N, -1))

        # 3. Residual acotado e inicializado en 0 sobre log1p(y_{t-1})
        delta = 0.55 * torch.tanh(self.head_delta(h_last).squeeze(-1))
        log_rate = torch.clamp(last_log_y + delta, min=-6.0, max=6.5)
        mu = torch.expm1(log_rate).clamp(min=1e-4)
        r = torch.exp(self.log_r).clamp(min=1e-2, max=1e3).unsqueeze(0).expand(B, N)
        return mu, r


def neg_binomial_nll(y, mu, r, eps=1e-6):
    """Negative Log-Likelihood exacta de la Binomial Negativa(mu, r)."""
    t1 = torch.lgamma(y + r) - torch.lgamma(r) - torch.lgamma(y + 1.0)
    t2 = r * torch.log(r / (r + mu + eps) + eps)
    t3 = y * torch.log(mu / (r + mu + eps) + eps)
    return -(t1 + t2 + t3).mean()

## 4. Entrenamiento con Early Stopping y registro en MLflow

In [ ]:
def compute_count_metrics(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0.0, None)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    rmsle = float(np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2)))
    wape = float(np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)))
    p = f"{prefix}_" if prefix else ""
    return {f"{p}mae": round(mae, 4), f"{p}rmse": round(rmse, 4), f"{p}rmsle": round(rmsle, 4), f"{p}wape": round(wape, 4)}

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    ys, mus, losses = [], [], []
    for x_b, y_b in loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        mu, r = model(x_b, A_norm_t)
        losses.append(float(neg_binomial_nll(y_b, mu, r).item()))
        ys.append(y_b.cpu().numpy())
        mus.append(mu.cpu().numpy())
    return float(np.mean(losses)), np.concatenate(ys, axis=0), np.concatenate(mus, axis=0)

HIDDEN_DIM = 64
LR = 5e-4
EPOCHS = 120
PATIENCE = 25

model = STGNNNegBin(in_feats=len(feature_cols), hidden_dim=HIDDEN_DIM, num_nodes=N_NODES, dropout=0.2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_rmsle = float("inf")
best_val_loss = float("inf")
best_state = None
wait = 0
history = {"train_nll": [], "val_nll": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_losses = []
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        mu, r = model(x_b, A_norm_t)
        loss = neg_binomial_nll(y_b, mu, r)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        tr_losses.append(float(loss.item()))

    scheduler.step()
    tr_nll = float(np.mean(tr_losses))
    va_nll, y_va, mu_va = predict_loader(model, val_loader)
    m_va = compute_count_metrics(y_va.ravel(), mu_va.ravel(), "val")
    history["train_nll"].append(tr_nll)
    history["val_nll"].append(va_nll)

    if m_va["val_rmsle"] < best_val_rmsle:
        best_val_rmsle = m_va["val_rmsle"]
        best_val_loss = va_nll
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping en época {epoch} | mejor val_rmsle={best_val_rmsle:.4f}")
            break

    if epoch % 10 == 0 or epoch == 1:
        print(f"Época {epoch:03d} | train_nll={tr_nll:.4f} | val_nll={va_nll:.4f} | val_mae={m_va['val_mae']} | val_rmsle={m_va['val_rmsle']}")

model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
va_nll, y_va, mu_va = predict_loader(model, val_loader)
te_nll, y_te, mu_te = predict_loader(model, test_loader)

val_metrics = compute_count_metrics(y_va.ravel(), mu_va.ravel(), "val")
test_metrics = compute_count_metrics(y_te.ravel(), mu_te.ravel(), "test")
print("\nST-GNN (Best) ->", val_metrics, test_metrics)

In [ ]:
# Curva de aprendizaje y predicciones en cantones 101 y 305
fig, axes = plt.subplots(3, 1, figsize=(13, 10))

axes[0].plot(history["train_nll"], label="Train NB-NLL")
axes[0].plot(history["val_nll"], label="Val NB-NLL")
axes[0].set_title("Curva de Entrenamiento ST-GNN (Negative Binomial NLL)")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("NB-NLL")
axes[0].grid(False)
axes[0].legend()

y_eval = np.concatenate([y_va, y_te], axis=0)
mu_eval = np.concatenate([mu_va, mu_te], axis=0)
t_eval = np.concatenate([va_t, te_t])

for ax, cid in zip(axes[1:], [101, 305]):
    c_idx = int(np.where(cantons == cid)[0][0])
    ax.plot(t_eval, y_eval[:, c_idx], color="black", lw=1.8, label="Casos reales")
    ax.plot(t_eval, mu_eval[:, c_idx], color="#1f77b4", lw=1.6, label="ST-GNN (NB-NLL)")
    ax.axvline(te_t[0], color="gray", ls=":", label="Inicio Test (2025)")
    ax.set_title(f"Pronóstico ST-GNN en Val (2024) y Test (2025) — Cantón {cid}")
    ax.set_ylabel("Casos")
    ax.set_ylim(0, 800)
    ax.legend(loc="upper right")
    ax.grid(False)

axes[-1].set_xlabel("time_idx")
fig.tight_layout()
fig_stgnn = ROOT_DIR / "data/processed/eda/10_nivel2_stgnn_predicciones.png"
fig.savefig(fig_stgnn, dpi=150)
plt.show()

En el segundo modelo se observa que ST-GNN tiene tendencia a comprender de mejor manera el comportamiento de los datos en el tiempo, hasta el momento ha sido el mejor modelo entre:

1) Arimax
2) HGB - Con FineTunnig
3) GLM - Binomial Negativa por Sobredispersión.

In [ ]:
ckpt_dir = ROOT_DIR / "models/checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / "stgnn_negbin_best.pt"
torch.save(best_state, ckpt_path)

mlruns_dir = (ROOT_DIR / "mlruns").resolve()
mlflow.set_tracking_uri(f"sqlite:///{(mlruns_dir / 'mlflow.db').as_posix()}")
mlflow.set_experiment("dengue_cr_spatiotemporal")

with mlflow.start_run(run_name="02_stgnn_gcn_gru_negbin"):
    mlflow.set_tags({"stage": "nivel2_stgnn", "graph_type": f"functional_corr_top{TOP_K}", "split_policy": "train<=2023|val=2024|test=2025"})
    mlflow.log_params({
        "window_size": WINDOW,
        "hidden_dim": HIDDEN_DIM,
        "lr": LR,
        "top_k_graph": TOP_K,
        "loss": "neg_binomial_nll",
        "n_features": len(feature_cols),
    })
    mlflow.log_metrics({"best_val_nll": round(best_val_loss, 4), **val_metrics, **test_metrics})
    mlflow.log_artifact(str(fig_stgnn), artifact_path="nivel2_plots")
    mlflow.log_artifact(str(ckpt_path), artifact_path="checkpoints")

exp_runs = mlflow.search_runs(experiment_names=["dengue_cr_spatiotemporal"])
exp_runs = exp_runs[exp_runs["tags.stage"] != "nivel1_tuning"]
cols_show = [c for c in ["tags.mlflow.runName", "tags.stage", "metrics.val_mae", "metrics.val_rmsle", "metrics.val_wape", "metrics.test_mae", "metrics.test_rmsle", "metrics.test_wape"] if c in exp_runs.columns]
display(exp_runs[cols_show].sort_values("metrics.val_rmsle", na_position="last"))